# Book recommendations from the Book-Crossing data lake

**Tanishk Khandelwal** · LIT2023020

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile
from urllib.request import urlretrieve
import csv
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from IPython.display import display

bronze, silver, gold = map(Path, ('bronze', 'silver', 'gold'))
for folder in (bronze, silver, gold):
    folder.mkdir(exist_ok=True)
files = {'users': 'BX-Users.csv', 'books': 'BX-Books.csv', 'ratings': 'BX-Book-Ratings.csv'}
archive = Path('book-crossings.zip')
source = 'https://cdn.freecodecamp.org/project-data/books/book-crossings.zip'
if not all(Path(f).exists() for f in files.values()) and not archive.exists():
    urlretrieve(source, archive)

## Bronze ingestion
The source files use Latin-1 and semicolons. The books file also uses backslash-escaped quotation marks inside titles. Reading every source column as text preserves ISBNs such as `0446672211` and the literal `NULL` markers for the next step.

In [2]:
from io import TextIOWrapper

def records(stream, name):
    reader = csv.reader(stream, delimiter=';', doublequote=True, escapechar=chr(92) if name == 'books' else None)
    headers = next(reader)
    return pd.DataFrame.from_records(reader, columns=headers)

stamp = pd.Timestamp.now(tz='UTC')
for name, filename in files.items():
    if Path(filename).exists():
        with open(filename, encoding='latin-1', newline='') as stream:
            table = records(stream, name)
    else:
        with ZipFile(archive) as z:
            member, = [p for p in z.namelist() if Path(p).name.casefold() == filename.casefold()]
            with z.open(member) as raw, TextIOWrapper(raw, encoding='latin-1', newline='') as stream:
                table = records(stream, name)
    table['ingestion_timestamp'] = stamp
    table.to_parquet(bronze / f'{name}.parquet', index=False)
del table

## Explore Bronze
**Shape, types and missing fields.** Bronze contains 278,858 users, 271,379 books and 1,149,780 rating rows. Source fields are strings, and 110,762 ages are missing. Count blanks and the source marker `NULL` as missing for this profile, then convert numeric fields and handle missing values in Silver.

In [3]:
users, books, ratings = [pd.read_parquet(bronze / f'{k}.parquet') for k in files]
def missing(s):
    return s.isna() | s.astype('string').str.strip().str.upper().isin(['', 'NULL'])
for name, df in zip(files, (users, books, ratings)):
    print(name, df.shape)
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.apply(lambda s: missing(s).sum())}))

users (278858, 4)


,dtype,missing
User-ID,object,0
Location,object,0
Age,object,110762
ingestion_timestamp,"datetime64[us, UTC]",0


books (271379, 9)


,dtype,missing
ISBN,object,0
Book-Title,object,0
Book-Author,object,0
Year-Of-Publication,object,0
Publisher,object,0
Image-URL-S,object,0
Image-URL-M,object,0
Image-URL-L,object,0
ingestion_timestamp,"datetime64[us, UTC]",0


ratings (1149780, 4)


,dtype,missing
User-ID,object,0
ISBN,object,0
Book-Rating,object,0
ingestion_timestamp,"datetime64[us, UTC]",0


**Rating distribution.** Of 1,149,780 ratings, 716,109 are zero and represent implicit feedback. Silver will keep explicit integer ratings from 1 to 10.

In [4]:
values = pd.to_numeric(ratings['Book-Rating'], errors='coerce')
display(values.value_counts(dropna=False).sort_index().rename('rows').to_frame())
print(f'Zero ratings: {values.eq(0).sum():,} of {len(values):,}')

,rows
Book-Rating,
0,716109
1,1770
2,2759
3,5996
4,8904
5,50974
6,36924
7,76457
8,103736


Zero ratings: 716,109 of 1,149,780


**Ages and publication years.** Ages range from 0 to 244, including 1,248 outside the allowed range. Publication years span 0 to 2050. Silver will impute ages from the valid-age median and leave unknown years null.

In [5]:
year_now = datetime.now(timezone.utc).year
for col, s, bounds in [('Age', users.Age, (5, 100)), ('Year', books['Year-Of-Publication'], (1, year_now))]:
    x = pd.to_numeric(s, errors='coerce')
    print(col, 'numeric values outside range:', (x.notna() & ~x.between(*bounds)).sum(),
          'missing or nonnumeric:', x.isna().sum())
    display(x.describe(percentiles=[.1,.25,.5,.75,.9]).to_frame(col))
    width = 10
    display((x // width * width).value_counts().sort_index().rename('rows by decade start').to_frame())

Age numeric values outside range: 1248 missing or nonnumeric: 110762


,Age
count,168096.000000
mean,34.751434
std,14.428097
min,0.000000
10%,19.000000
25%,24.000000
50%,32.000000
75%,44.000000
90%,55.000000
max,244.000000


,rows by decade start
Age,
0.0,1069
10.0,18894
20.0,51539
30.0,41889
40.0,26700
50.0,18811
60.0,6916
70.0,1644
80.0,199


Year numeric values outside range: 4630 missing or nonnumeric: 0


,Year
count,271379.000000
mean,1959.756050
std,258.011363
min,0.000000
10%,1982.000000
25%,1989.000000
50%,1995.000000
75%,2000.000000
90%,2002.000000
max,2050.000000


,rows by decade start
Year-Of-Publication,
0,4619
1370,2
1800,1
1890,1
1900,17
1910,23
1920,65
1930,57
1940,127


**Ratings per user and book.** The median observed user and ISBN each have only one rating. A minimum of 10 explicit ratings trades coverage for more observations per item and user when estimating similarities. The support summary quantifies the coverage cost before repeated pruning.

In [6]:
minimum = 10
for col in ['User-ID', 'ISBN']:
    counts = ratings.groupby(col).size()
    explicit = ratings.loc[values.between(1,10), [col, 'ISBN' if col == 'User-ID' else 'User-ID']].drop_duplicates()
    support = explicit.groupby(col).size().reindex(counts.index, fill_value=0)
    display(counts.describe(percentiles=[.5,.75,.9,.99]).to_frame('all ratings per '+col))
    print(f'{col}: {support.lt(minimum).sum():,} of {len(support):,} observed IDs have fewer than {minimum} explicit pairs.')

,all ratings per User-ID
count,105283.000000
mean,10.920851
std,90.562825
min,1.000000
50%,1.000000
75%,4.000000
90%,12.000000
99%,174.000000
max,13602.000000


User-ID: 97,949 of 105,283 observed IDs have fewer than 10 explicit pairs.


,all ratings per ISBN
count,340556.000000
mean,3.376185
std,12.436252
min,1.000000
50%,1.000000
75%,2.000000
90%,6.000000
99%,36.000000
max,2502.000000


ISBN: 334,911 of 340,556 observed IDs have fewer than 10 explicit pairs.


**Matrix sparsity.** Only 0.00136% of catalog user-book pairs have a recorded interaction, leaving 99.99864% empty. After Silver checks that both IDs exist, Gold will store only observed explicit ratings in a sparse matrix.

In [7]:
eligible = ratings['User-ID'].isin(users['User-ID']) & ratings.ISBN.isin(books.ISBN)
possible = users['User-ID'].nunique() * books.ISBN.nunique()
observed = ratings.loc[eligible, ['User-ID','ISBN']].drop_duplicates().shape[0]
density = 100 * observed / possible
print(f'{observed:,} / {possible:,} possible pairs observed ({density:.5f}% coverage, {100-density:.5f}% empty).')

1,031,175 / 75,676,205,182 possible pairs observed (0.00136% coverage, 99.99864% empty).


## Silver cleaning
A blank age and an age of 244 both need replacing. The median comes from ages between 5 and 100 before filtering out infrequent readers. Users and books with fewer than ten explicit ratings are removed together. Removing a book can leave a previously eligible user with nine ratings, so the count is checked again after every pass.

In [8]:
def clean_base(df):
    df = df.drop(columns='ingestion_timestamp')
    return df.drop_duplicates().copy()

def clean_text(s):
    s = s.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    return s.mask(s.str.upper().isin(['NULL', '']))

def user_ids(s):
    return pd.to_numeric(s.where(s.astype('string').str.fullmatch(r'[0-9]+', na=False)), errors='coerce').astype('Int64')

u = clean_base(users)
u['User-ID'] = user_ids(u['User-ID'])
u = u.dropna(subset=['User-ID']).drop_duplicates('User-ID').copy()
ages = pd.to_numeric(u.Age, errors='coerce')
valid_ages = ages.where(ages.between(5,100))
u['Age'] = valid_ages.fillna(valid_ages.median())
parts = clean_text(u.pop('Location')).str.lower().str.rsplit(',', n=2)
for i, col in enumerate(['city','state','country']):
    u[col] = clean_text(parts.str.get(i))

b = clean_base(books).drop(columns=books.filter(regex='^Image-URL').columns)
b['ISBN'] = clean_text(b.ISBN).str.upper()
for col in ['Book-Title','Book-Author','Publisher']:
    b[col] = clean_text(b[col]).str.lower()
years = pd.to_numeric(b['Year-Of-Publication'], errors='coerce')
b['Year-Of-Publication'] = years.where(years.between(1,year_now) & years.mod(1).eq(0)).astype('Int64')
b = b.dropna(subset=['ISBN','Book-Title']).drop_duplicates('ISBN')
r = clean_base(ratings)
r['User-ID'] = user_ids(r['User-ID'])
r['ISBN'] = clean_text(r.ISBN).str.upper()
r['Book-Rating'] = pd.to_numeric(r['Book-Rating'], errors='coerce')
r = r[r['Book-Rating'].isin(range(1,11)) & r['User-ID'].isin(u['User-ID']) & r.ISBN.isin(b.ISBN)]
r = r.drop_duplicates().groupby(['User-ID','ISBN'], as_index=False)['Book-Rating'].mean()
before_support = len(r)
while len(r):
    keep = (r.groupby('User-ID').ISBN.transform('size') >= minimum) & (r.groupby('ISBN').ISBN.transform('size') >= minimum)
    if keep.all():
        break
    r = r.loc[keep]
if r.empty:
    raise ValueError('No ratings remain at the selected support threshold.')
print(f'Support filter kept {len(r):,} of {before_support:,} linked explicit pairs ({len(r)/before_support:.1%}).')
u = u[u['User-ID'].isin(r['User-ID'])]
b = b[b.ISBN.isin(r.ISBN)]
for name, df in zip(files, (u,b,r)):
    df.to_parquet(silver / f'{name}.parquet', index=False)
    print(name, df.shape)

# Keep previously encountered titles, including zero-rated books, for serving.
title_map = pd.Series(clean_text(books['Book-Title']).str.lower().array,
                      index=clean_text(books.ISBN).str.upper())
title_map = title_map[~title_map.index.duplicated()]
history = pd.DataFrame({'user': user_ids(ratings['User-ID']),
                        'ISBN': clean_text(ratings.ISBN).str.upper()})
history = history.loc[history.user.isin(u['User-ID'])].drop_duplicates().copy()
history['title'] = history.ISBN.map(title_map)
history.dropna().to_parquet(silver/'reading_history.parquet', index=False)

Support filter kept 41,469 of 383,853 linked explicit pairs (10.8%).
users (1820, 5)
books (2030, 5)
ratings (41469, 3)


## Gold model inputs
Persist a user-by-book matrix and the ordered identifiers that define its axes. Aggregates describe the same explicit ratings used in training.

In [9]:
r = pd.read_parquet(silver/'ratings.parquet')
b = pd.read_parquet(silver/'books.parquet')
uid = pd.Index(sorted(r['User-ID'].unique()), name='User-ID')
isbn = pd.Index(sorted(r.ISBN.unique()), name='ISBN')
matrix = sparse.csr_matrix((r['Book-Rating'].to_numpy(dtype='float32'), (uid.get_indexer(r['User-ID']), isbn.get_indexer(r.ISBN))), shape=(len(uid),len(isbn)))
sparse.save_npz(gold/'user_book.npz', matrix)
uid.to_frame(index=False).to_parquet(gold/'users.parquet', index=False)
b.set_index('ISBN').loc[isbn,['Book-Title','Book-Author']].reset_index().to_parquet(gold/'books.parquet', index=False)
for key, name in [('User-ID','user'), ('ISBN','book')]:
    agg = r.groupby(key)['Book-Rating'].agg(average_rating='mean', rating_count='size')
    agg.reset_index().to_parquet(gold/f'{name}_aggregates.parquet', index=False)
history = pd.read_parquet(silver/'reading_history.parquet')
history.loc[history.title.isin(b['Book-Title'])].to_parquet(gold/'reading_history.parquet', index=False)

## Item-based recommendations
Cosine similarity compares books across their shared user space. For repeated titles, use the edition with the most explicit ratings. For a user, combine up to 50 neighbors per seed rated at least 8/10. Rank candidates by the sum of similarity times seed rating. Exclude all previously recorded books, including implicit interactions and other editions sharing a title.

In [10]:
matrix = sparse.load_npz(gold/'user_book.npz').tocsr()
items = matrix.T.tocsr()
catalog = pd.read_parquet(gold/'books.parquet')
uid = pd.Index(pd.read_parquet(gold/'users.parquet')['User-ID'])
book_rating_counts = np.diff(items.indptr)
history = pd.read_parquet(gold/'reading_history.parquet')
seen_by_user = history.groupby('user').title.agg(set).to_dict()
titles = catalog['Book-Title'].to_numpy()

model = NearestNeighbors(metric='cosine', algorithm='brute').fit(items)

def neighbors(i, k):
    count = min(k + 1, items.shape[0])
    distances, indices = model.kneighbors(items[i], n_neighbors=count)
    return [(j, max(0., 1. - distance))
            for j, distance in zip(indices[0], distances[0]) if j != i][:k]

def similar_books(title):
    key = ' '.join(title.lower().split())
    matches = np.flatnonzero(catalog['Book-Title'].eq(key).to_numpy(dtype=bool))
    if not len(matches):
        raise KeyError(title)
    i = matches[np.argmax(book_rating_counts[matches])]
    pairs = neighbors(i, items.shape[0])
    result = catalog.iloc[[j for j,s in pairs]].assign(similarity=[s for j,s in pairs])
    return result.loc[result['Book-Title'].ne(key)].drop_duplicates('Book-Title').head(10).reset_index(drop=True)

def seeds(user_id):
    row = matrix.getrow(uid.get_loc(user_id))
    return [(j,float(v)) for j,v in zip(row.indices,row.data) if v >= 8]

def recommend(user_id, n=10):
    seen = seen_by_user[user_id]
    scores = {}
    for i, rating in seeds(user_id):
        for j, similarity in neighbors(i, 50):
            if similarity > 0 and titles[j] not in seen:
                scores[j] = scores.get(j, 0.) + rating * similarity
    ranked = sorted(scores, key=lambda j: (-scores[j], j))
    result = catalog.iloc[ranked].assign(score=[scores[j] for j in ranked])
    return result.drop_duplicates('Book-Title').head(max(0, n)).reset_index(drop=True)


## Demonstration and checks
Display three users with high-rated seeds and ten available suggestions. The two lists in each display are independently ranked. Scores combine evidence from all of that user's seeds.

In [11]:
assert matrix.nnz == len(r)
assert min(np.diff(matrix.indptr)) >= minimum and min(book_rating_counts) >= minimum
query = catalog.loc[book_rating_counts.argmax(), 'Book-Title']
print(f'Ten titles similar to "{query}"')
similar = similar_books(query)
assert len(similar) == similar['Book-Title'].nunique() == 10
display(similar)
shown = 0
for user_id in uid:
    high = seeds(user_id)
    if not high:
        continue
    recs = recommend(user_id)
    if len(recs) < 10:
        continue
    high.sort(key=lambda p: (-p[1],p[0]))
    liked = pd.DataFrame([(catalog.at[j,'Book-Title'],v) for j,v in high], columns=['Highly rated book','Rating'])
    assert len(recs) == recs['Book-Title'].nunique() == 10
    assert set(recs['Book-Title']).isdisjoint(seen_by_user[user_id])
    assert recs.score.is_monotonic_decreasing
    print(f'User {user_id}')
    display(pd.concat([liked, recs[['Book-Title','score']].rename(columns={'Book-Title':'Recommended book'})], axis=1).fillna(''))
    shown += 1
    if shown == 3:
        break
assert shown == 3, 'Fewer than three users qualified for the ten-title demonstration.'

Ten titles similar to "the lovely bones: a novel"


,ISBN,Book-Title,Book-Author,similarity
0,0446672211,where the heart is (oprah's book club (paperback)),billie letts,0.219241
1,0316096199,lucky : a memoir,alice sebold,0.217959
2,0385504209,the da vinci code,dan brown,0.178328
3,0316601950,the pilot's wife : a novel,anita shreve,0.176339
4,0743418174,good in bed,jennifer weiner,0.175819
5,0446531332,nights in rodanthe,nicholas sparks,0.170172
6,0688170528,the pact: a love story,jodi picoult,0.168194
7,0316603287,the lake house,james patterson,0.166218
8,0684857820,lucky,alice sebold,0.165055
9,0316284955,white oleander : a novel (oprah's book club),janet fitch,0.154238


User 243


,Highly rated book,Rating,Recommended book,score
0,the bean trees,10.0,the book of ruth (oprah's book club (paperback)),3.847579
1,memoirs of a geisha,10.0,the body farm,3.575107
2,the pilot's wife : a novel,9.0,from potter's field,3.338379
3,me talk pretty one day,9.0,animal dreams,3.230984
4,unnatural exposure,9.0,point of origin,3.024232
5,the general's daughter,9.0,tears of the giraffe (no.1 ladies detective agency),2.957377
6,,,pigs in heaven,2.923849
7,,,silent prey,2.896350
8,,,the sisterhood of the traveling pants,2.875071
9,,,serpent's tooth : a peter decker/rina lazarus novel (peter decker &amp; rina lazarus novels (paperback)),2.776450


User 254


,Highly rated book,Rating,Recommended book,score
0,american gods,10.0,harry potter and the sorcerer's stone (harry potter (paperback)),12.357403
1,smoke and mirrors: short fictions and illusions,9.0,prodigal summer,8.340963
2,the secret life of bees,9.0,the secret garden,7.069443
3,american gods: a novel,9.0,charlotte's web (trophy newbery),6.235172
4,the bonesetter's daughter,9.0,the silmarillion,6.104952
5,harry potter and the chamber of secrets (book 2),9.0,speak,5.819704
6,harry potter and the chamber of secrets (book 2),9.0,the name of the rose: including postscript to the name of the rose,5.698191
7,harry potter and the prisoner of azkaban (book 3),9.0,brave new world,5.64916
8,harry potter and the prisoner of azkaban (book 3),9.0,"the reptile room (a series of unfortunate events, book 2)",5.570804
9,harry potter and the goblet of fire (book 4),9.0,slaughterhouse five or the children's crusade: a duty dance with death,5.449608


User 638


,Highly rated book,Rating,Recommended book,score
0,the lovely bones: a novel,10.0,midnight bayou,15.879581
1,four blind mice,10.0,the gift,10.867444
2,the catcher in the rye,10.0,trading places,10.777965
3,me talk pretty one day,10.0,dead aim,10.773626
4,barrel fever : stories and essays (barrel fever),10.0,the conspiracy club,10.433276
5,the pilot's wife : a novel tag: author of the weight of water (oprah's book club (hardcover)),10.0,the penwyth curse,9.903643
6,suzanne's diary for nicholas,10.0,someone to watch over me : a novel,9.574826
7,"the two towers (the lord of the rings, part 2)",10.0,blow fly: a scarpetta novel,9.480252
8,the da vinci code,10.0,the wedding,9.145685
9,skipping christmas,10.0,the face,8.923804
